In [1]:
import multiprocessing
import os

try:
    cores = int(os.environ["SLURM_JOB_CPUS_PER_NODE"])
except:
    cores = multiprocessing.cpu_count()


os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import batman
import blackjax
from tqdm import tqdm
import gpjax as gpx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from gpjax.kernels import RBF, Linear, Periodic, Polynomial
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import (
    Normal,
    TruncatedNormal,
    TransformedDistribution,
    Uniform,
)

from inference import log_likelihood_function
from kernels import OrnsteinUhlenbeck
from kernelsearch import KernelSearch, describe_kernel, get_trainables, set_trainables
from mcmc import nuts_warmup, run_mcmc
from util import calculate_example_lightcurve, lc_model

jax.config.update("jax_enable_x64", True)

rng_key = jax.random.PRNGKey(42)

Things to consider:
- heteroskedastic noise (supported natively -- check, or maybe not with fitting.. see if that important)
- doing kernel search per time series, or across all time series (loop over all datasets and calculate cumulative likelihood -> cummulative aic/bic)
- (normalise mean?)
- need longer time series data for proper learning

Questions:
- longer time series
- limb darkening fitting

## Load Data

In [2]:
df = pd.read_csv("data/lc.dat", sep=" ", header=None)
x = df[0].to_numpy()
x_min = np.amin(x)
x -= x_min
y = df.iloc[:, 1::2].to_numpy().T
yerr = df.iloc[:, 2::2].to_numpy().T

mask = np.ones_like(x, dtype=bool)
mask[100:324] = False

ld_pars = [
    (0.87, -0.0639),
    (0.70, -0.0109),
    (0.72, -0.0275),
    (0.64, 0.0460),
    (0.58, 0.0736),
    (0.53, 0.1032),
    (0.55, 0.0884),
    (0.51, 0.1080),
    (0.46, 0.1134),
    (0.45, 0.1265),
    (0.41, 0.1348),
    (0.43, 0.1269),
    (0.42, 0.1315),
    (0.41, 0.1334),
    (0.41, 0.1337),
    (0.36, 0.1404),
    (0.32, 0.1565),
    (0.35, 0.1429),
    (0.35, 0.1434),
]

## Get GP models for white light curve

In [3]:
kernel_library = [
    Linear(),
    RBF(),
    OrnsteinUhlenbeck(),
    Periodic(),
]

In [4]:
tree = KernelSearch(
    kernel_library,
    X=jnp.array(x[mask]),
    y=jnp.array(y[-1][mask]),
    obs_stddev=jnp.amax(yerr[-1][mask]),
    verbosity=1,
)

# model = tree.search(
#     depth=7,
#     n_leafs=4,
#     patience=1,
# )
# print(describe_kernel(model))

In [5]:
import pickle

model_name = "lc1_gpmodel"
# with open(f"saved/{model_name}", "wb") as file:
#     pickle.dump(model, file)
with open(f"saved/{model_name}", "rb") as file:
    model = pickle.load(file)
describe_kernel(model)

'OU + ((Linear * Linear * Linear) * Linear * RBF)'

## Fit white curve parameter

In [6]:
def white_lc_model(t, params):
    central = orbits.keplerian.Central(
        mass=params[0],
        radius=params[1],
    )

    # The light curve calculation requires an orbit
    orbit = orbits.keplerian.Body(
        central=central,
        period=3.9501907,
        radius=params[5] * central.radius,
        inclination=jnp.deg2rad(params[2]),
        time_transit=params[3],
    )

    lc = LimbDarkLightCurve([params[4], 0.862]).light_curve(orbit, t=t)
    return lc


log_likelihood = log_likelihood_function(
    model.unconstrain(),
    white_lc_model,
    x,
    y[-1],
    mask,
    fix_gp=False,
    compile=True,
)

lbfgsb = ScipyMinimize(fun=jit(lambda par: -log_likelihood(par)), method="l-bfgs-b")
lbfgsb_sol = lbfgsb.run(
    {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([1.45, 1.653, 89.3, 0.18, 0.53, 0.11]),
    }
)
white_lc_sol = lbfgsb_sol.params["lc_parameter"]

## Define LC model

In [7]:
def get_lc_model(ld_pars):
    def lc_model(t, params):
        central = orbits.keplerian.Central(
            mass=white_lc_sol[0],
            radius=white_lc_sol[1],
        )

        # The light curve calculation requires an orbit
        orbit = orbits.keplerian.Body(
            central=central,
            period=3.9501907,
            radius=params[0] * central.radius,
            inclination=jnp.deg2rad(white_lc_sol[2]),
            time_transit=white_lc_sol[3],
        )

        lc = LimbDarkLightCurve([params[1], ld_pars[1]]).light_curve(orbit, t=t)
        return lc

    return lc_model

## Define likelihood, prior, posterior

In [8]:
def get_logprob(model, y, lc_model, ld_pars):
    initial_position = {
        "gp_parameter": get_trainables(model, unconstrain=True),
        "lc_parameter": jnp.array([0.10, ld_pars[0]]),
    }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.06, 0.04],
        ),
    }

    log_likelihood = log_likelihood_function(
        model.unconstrain(),
        lc_model,
        x,
        y,
        mask,
        fix_gp=False,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## Fits

In [ ]:
sols = []
for i in tqdm(range(len(y) - 1)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        lc_model,
        ld_pars[i],
    )

    lbfgsb = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    )
    lbfgsb_sol = lbfgsb.run(initial_position)
    sol = lbfgsb_sol.params
    sols.append(sol)

In [ ]:
ref = np.array(
    [
        0.10790,
        0.10965,
        0.10831,
        0.10774,
        0.10766,
        0.10740,
        0.10709,
        0.10636,
        0.10724,
        0.10734,
        0.10757,
        0.10975,
        0.10810,
        0.10741,
        0.10700,
        0.10745,
        0.10725,
        0.10710,
        0.10710,
    ]
)

# plt.scatter(range(len(y) - 1), [sol["lc_parameter"][0] for sol in sols])
# plt.scatter(range(len(y) - 1), ref)

# plt.figure()
# plt.scatter(range(len(y) - 1), (ref - [sol["lc_parameter"][0] for sol in sols]) / ref)

## Do MCMC

In [20]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 200
num_samples = 150
num_chains = cores

rng_key, warmup_key = jax.random.split(rng_key, 2)

# run nuts adaption on white lc
lc_model = jit(get_lc_model((0.5326, 0.0862)))
log_probability, initial_position = get_logprob(
    model,
    y[-1],
    lc_model,
    ld_pars[-1],
)

state, parameters = nuts_warmup(
    warmup_key,
    log_probability,
    initial_position,
    num_steps=num_adapt,
)

chains = {"gp_parameter": [], "lc_parameter": []}

for i in tqdm(range(len(y) - 1)):
    lc_model = jit(get_lc_model(ld_pars[i]))
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        lc_model,
        ld_pars[i],
    )

    rng_key, sample_key = jax.random.split(rng_key, 2)

    initial_positions = {
        "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
        "lc_parameter": jnp.tile(state.position["lc_parameter"], (num_chains, 1)),
    }

    final_state, state_history, info_history = run_mcmc(
        sample_key,
        log_probability,
        parameters,
        initial_positions,
        num_steps=num_samples,
    )

    for par in ["gp_parameter", "lc_parameter"]:
        chain = np.array(
            state_history.position[par].reshape(
                -1, state_history.position[par].shape[-1]
            )
        )
        chains[par].append(chain)

np.savez(f"saved/{model_name}_parameter.npz", **chains)

Running window adaptation


100%|██████████| 2/2 [00:34<00:00, 17.02s/it]
